In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

c:\Users\roger\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# 1. Define the Hardware-Friendly CNN Architecture
class ZedBoardMNIST(nn.Module):
    def __init__(self):
        super(ZedBoardMNIST, self).__init__()
        # Executed on FPGA PL (Programmable Logic) via AXI-DMA
        # Input: 1 x 28 x 28 -> Output: 8 x 26 x 26
        # bias=False simplifies FPGA MAC hardware
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=0, bias=False)
        
        # Executed on ARM PS (Processing System) in PetaLinux (or in FPGA prior to S2MM)
        # ReLU + 2x2 MaxPool turns 8 x 26 x 26 -> 8 x 13 x 13 (1352 elements)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected layers on ARM PS
        self.fc1 = nn.Linear(8 * 13 * 13, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        # --- FPGA PL Stage ---
        x = self.conv1(x)
        
        # --- ARM PS (or trailing PL stream) Stage ---
        x = F.relu(x)
        x = self.pool(x)
        
        # Flatten for FC layers: (Batch, 8*13*13)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [3]:
def export_for_zedboard(model, export_dir="zedboard_weights"):
    os.makedirs(export_dir, exist_ok=True)
    model.eval()
    
    print(f"\nExporting weights to '{export_dir}/'...")
    
    # Save standard PyTorch state dict
    torch.save(model.state_dict(), os.path.join(export_dir, "mnist_cnn.pth"))
    
    # Extract numpy arrays (Float32)
    conv1_w = model.conv1.weight.detach().cpu().numpy().astype(np.float32) # Shape: (8, 1, 3, 3)
    fc1_w   = model.fc1.weight.detach().cpu().numpy().astype(np.float32)   # Shape: (32, 1352)
    fc1_b   = model.fc1.bias.detach().cpu().numpy().astype(np.float32)     # Shape: (32,)
    fc2_w   = model.fc2.weight.detach().cpu().numpy().astype(np.float32)   # Shape: (10, 32)
    fc2_b   = model.fc2.bias.detach().cpu().numpy().astype(np.float32)     # Shape: (10,)

    # 1. Save as .npy (if using Python on PetaLinux)
    np.save(os.path.join(export_dir, "conv1_w.npy"), conv1_w)
    np.save(os.path.join(export_dir, "fc1_w.npy"), fc1_w)
    np.save(os.path.join(export_dir, "fc1_b.npy"), fc1_b)
    np.save(os.path.join(export_dir, "fc2_w.npy"), fc2_w)
    np.save(os.path.join(export_dir, "fc2_b.npy"), fc2_b)

    # 2. Save as raw flat binary .bin (if loading via C/C++ fread on PetaLinux)
    conv1_w.tofile(os.path.join(export_dir, "conv1_w.bin"))
    fc1_w.tofile(os.path.join(export_dir, "fc1_w.bin"))
    fc1_b.tofile(os.path.join(export_dir, "fc1_b.bin"))
    fc2_w.tofile(os.path.join(export_dir, "fc2_w.bin"))
    fc2_b.tofile(os.path.join(export_dir, "fc2_b.bin"))

    # 3. Generate a C header file with 8-bit Quantized Conv weights (Optional for fixed-point Verilog IP)
    # Scales conv1 weights to int8 [-127, 127]
    max_val = np.max(np.abs(conv1_w))
    scale_factor = 127.0 / max_val if max_val > 0 else 1.0
    conv1_w_int8 = np.round(conv1_w * scale_factor).astype(np.int8)
    
    header_path = os.path.join(export_dir, "conv_weights_int8.h")
    with open(header_path, "w") as f:
        f.write("// Auto-generated 8-bit quantized weights for FPGA Conv1 Layer\n")
        f.write(f"// Dequantization scale factor: {scale_factor:.6f}f\n\n")
        f.write("#ifndef CONV_WEIGHTS_H\n#define CONV_WEIGHTS_H\n\n")
        f.write("#include <stdint.h>\n\n")
        f.write(f"const float CONV1_SCALE = {scale_factor:.6f}f;\n")
        f.write("const int8_t conv1_weights[8][9] = {\n")
        for filt_idx in range(8):
            flat_kernel = conv1_w_int8[filt_idx, 0].flatten()
            vals_str = ", ".join(f"{val:4d}" for val in flat_kernel)
            f.write(f"    {{ {vals_str} }},\n")
        f.write("};\n\n#endif // CONV_WEIGHTS_H\n")

    print("Export complete! Shapes exported:")
    print(f"  - conv1_w : {conv1_w.shape} (8 filters, 3x3 kernel)")
    print(f"  - fc1_w   : {fc1_w.shape}")
    print(f"  - fc1_b   : {fc1_b.shape}")
    print(f"  - fc2_w   : {fc2_w.shape}")
    print(f"  - fc2_b   : {fc2_b.shape}")

In [4]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    # ToTensor() scales uint8 [0, 255] pixels to float32 [0.0, 1.0].
    # We deliberately omit Normalize() so the Raspberry Pi doesn't have to normalize frames.
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    model = ZedBoardMNIST().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 5
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Evaluate after each epoch
        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        acc = 100. * correct / len(test_dataset)
        print(f"Epoch {epoch}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Test Accuracy: {acc:.2f}%")

    export_for_zedboard(model)

if __name__ == "__main__":
    main()

Training on device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 11.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 19.8MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.06MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.96MB/s]


Epoch 1/5 - Loss: 0.3747 - Test Accuracy: 94.88%
Epoch 2/5 - Loss: 0.1434 - Test Accuracy: 96.71%
Epoch 3/5 - Loss: 0.1005 - Test Accuracy: 97.33%
Epoch 4/5 - Loss: 0.0809 - Test Accuracy: 97.64%
Epoch 5/5 - Loss: 0.0683 - Test Accuracy: 97.59%

Exporting weights to 'zedboard_weights/'...
Export complete! Shapes exported:
  - conv1_w : (8, 1, 3, 3) (8 filters, 3x3 kernel)
  - fc1_w   : (32, 1352)
  - fc1_b   : (32,)
  - fc2_w   : (10, 32)
  - fc2_b   : (10,)
